# Session 2 — DGAT model and pretrained predictions

Prepare graph inputs, inspect the DGAT architecture and objective without full training, then load and visualize the verified pretrained predictions.

For workshop reproducibility, select **Runtime → Change runtime type → 2026.04**
(Python 3.12) before running the bootstrap.

Work through the parts in order. Outputs and JSON checkpoints are written to
`MyDrive/ECCB2026/state`, so they survive a Colab runtime reset. If the runtime stops,
rerun the bootstrap cell, inspect the printed completed checkpoints, and jump to the
first unfinished part; each part reloads its required inputs.


In [ ]:
from pathlib import Path
import importlib
import importlib.util
import json
import os
import shutil
import subprocess
import sys

SESSION_REQUIREMENTS = [('anndata', 'anndata==0.11.4'), ('scanpy', 'scanpy==1.11.5'), ('muon', 'muon==0.1.7')]
NEED_DGAT = True

in_colab = importlib.util.find_spec("google.colab") is not None and Path("/content").is_dir()
if in_colab:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    repo_dir = Path("/content/ECCB-2026-Tutorial")
    tutorial_root = repo_dir / "hands-on_tutorial"
    if not (tutorial_root / "src" / "dgat_tutorial").is_dir():
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/osmanbeyoglulab/ECCB-2026-Tutorial.git", str(repo_dir)],
            check=True,
        )
    else:
        # A Colab runtime can outlive the notebook tab. Refresh an existing clone
        # so newly opened notebooks do not silently execute stale setup scripts.
        subprocess.run(
            ["git", "-C", str(repo_dir), "fetch", "--depth", "1", "origin", "main"],
            check=True,
        )
        subprocess.run(
            ["git", "-C", str(repo_dir), "reset", "--hard", "origin/main"],
            check=True,
        )

    drive_root = Path("/content/drive/MyDrive/ECCB2026")
    drive_data = drive_root / "assets" / "DGAT_assets" / "data"
    manifest_path = drive_root / "asset_manifest.json"
    if not manifest_path.is_file():
        raise FileNotFoundError(f"Missing {manifest_path}. Run Session 0 before the tutorial.")
    asset_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    local_data = tutorial_root / "external" / "DGAT_assets" / "data"
    local_data.mkdir(parents=True, exist_ok=True)
    for filename in ("Tonsil_RNA.h5ad", "Tonsil_ADT.h5ad"):
        source = drive_data / filename
        destination = local_data / filename
        if not source.is_file() or source.stat().st_size == 0:
            raise FileNotFoundError(
                f"Missing {source}. Run Session 0 Drive preparation before the tutorial."
            )
        expected_bytes = asset_manifest["files"][filename]["bytes"]
        if source.stat().st_size != expected_bytes:
            raise IOError(
                f"Drive asset size mismatch for {filename}: expected {expected_bytes}, "
                f"found {source.stat().st_size}. Rerun Session 0."
            )
        if not destination.is_file() or destination.stat().st_size != source.stat().st_size:
            print(f"Copying {filename} from Drive to the Colab VM ...")
            shutil.copy2(source, destination)

    os.environ["DGAT_TUTORIAL_STATE_DIR"] = str(drive_root / "state")

    if NEED_DGAT:
        dgat_dir = tutorial_root / "external" / "DGAT"
        if not (dgat_dir / "utils" / "Preprocessing.py").is_file():
            dgat_dir.parent.mkdir(parents=True, exist_ok=True)
            subprocess.run(
                ["git", "clone", "--depth", "1",
                 "https://github.com/osmanbeyoglulab/DGAT.git", str(dgat_dir)],
                check=True,
            )

    missing_specs = [
        package_spec
        for import_name, package_spec in SESSION_REQUIREMENTS
        if importlib.util.find_spec(import_name) is None
    ]
    if missing_specs:
        wheelhouse = drive_root / "wheelhouse" / f"py{sys.version_info.major}{sys.version_info.minor}"
        online_command = [sys.executable, "-m", "pip", "install", "-q", *missing_specs]
        if wheelhouse.is_dir() and any(wheelhouse.glob("*.whl")):
            print(f"Installing missing packages using the Drive wheel cache: {missing_specs}")
            cached_command = [
                sys.executable, "-m", "pip", "install", "-q", "--no-index",
                "--find-links", str(wheelhouse), *missing_specs,
            ]
            try:
                subprocess.run(cached_command, check=True)
            except subprocess.CalledProcessError:
                print("Wheel cache was incomplete; falling back to PyPI.")
                subprocess.run(online_command, check=True)
        else:
            print(f"Drive wheel cache missing; installing from PyPI: {missing_specs}")
            subprocess.run(online_command, check=True)
        importlib.invalidate_caches()
else:
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for candidate in candidates:
        if (candidate / "src" / "dgat_tutorial").is_dir():
            tutorial_root = candidate
            break
    else:
        raise FileNotFoundError("Could not locate hands-on_tutorial/ from the current directory.")

os.chdir(tutorial_root)
src_dir = tutorial_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from dgat_tutorial.checkpoints import tutorial_paths, write_checkpoint

paths = tutorial_paths(tutorial_root)
completed = sorted(path.name for path in paths.checkpoints.glob("session_*/part_*.json"))
print(f"Tutorial root: {paths.root}")
print(f"Persistent state: {paths.checkpoints.parent}")
print("Completed checkpoints:", completed or "none yet")


## Session 2 · Part 1 — Prepare paired graphs for DGAT

**Goal:** turn aligned, normalized RNA and protein measurements into the two graph inputs used during
training. A spatial transcriptomics prediction sample has RNA only; paired spatial CITE-seq training
samples provide both modalities. This notebook uses the same official Tonsil RNA/ADT pair as Session 1,
so the tensor dimensions and graph construction correspond to the real tutorial dataset.


### 1. Process paired modalities


In [ ]:
import numpy as np
import pandas as pd

from dgat_tutorial.data import find_dgat_h5ad_pair, load_tutorial_data
from dgat_tutorial.processing import build_dgat_graphs, knn_edge_index, process_modalities_official_dgat

pair = find_dgat_h5ad_pair(paths.raw_data)
dataset = load_tutorial_data(paths.raw_data)
processed = process_modalities_official_dgat(
    dataset.spots,
    dataset.transcripts,
    dataset.proteins,
    dgat_repo_dir=paths.root / "external" / "DGAT",
)
source = f"RNA={pair[0]}, ADT={pair[1]}"


### 2. Construct and save the RNA and protein graph inputs


In [ ]:
# The official pipeline stores these as PyTorch Geometric HeteroData node/edge types.
# Here we expose the arrays first so dimensions and alignment are easy to inspect.
x_rna = processed.normalized_transcripts.to_numpy(dtype=np.float32)
x_protein = processed.normalized_proteins.to_numpy(dtype=np.float32)
graphs = build_dgat_graphs(
    processed.spots,
    processed.normalized_transcripts,
    processed.normalized_proteins,
)
rna_edge_index = graphs["rna_edge_index"]
protein_edge_index = graphs["protein_edge_index"]
spatial_edge_index = graphs["spatial_edge_index"]
print(
    "DGAT graphs = spatial 6-NN ∪ molecular 10-NN "
    f"(RNA PCA if >1500 genes). "
    f"spatial={spatial_edge_index.shape[1]}, "
    f"rna_union={rna_edge_index.shape[1]}, "
    f"protein_union={protein_edge_index.shape[1]}"
)

graph_summary = pd.DataFrame([
    {"graph": "RNA", "nodes": len(x_rna), "node_features": x_rna.shape[1], "directed_edges": rna_edge_index.shape[1]},
    {"graph": "protein", "nodes": len(x_protein), "node_features": x_protein.shape[1], "directed_edges": protein_edge_index.shape[1]},
])
display(graph_summary)

# Save the node-order mapping, graph summary, RNA union edges, and checkpoint metadata.
ids_path = paths.processed_data / "aligned_spot_ids.csv"
summary_path = paths.results / "session02_graph_input_summary.csv"
edge_path = paths.processed_data / "rna_spatial_edge_index.csv"
pd.DataFrame({"spot_id": processed.spots.index}).to_csv(ids_path, index=False)
graph_summary.assign(source=source).to_csv(summary_path, index=False)
pd.DataFrame(rna_edge_index.T, columns=["source_index", "target_index"]).to_csv(edge_path, index=False)
manifest = write_checkpoint(
    "2.1", [ids_path, summary_path, edge_path],
    summary={"source": source, "spots": len(processed.spots), "genes": x_rna.shape[1], "proteins": x_protein.shape[1]},
    start=paths.root,
)
print(f"Checkpoint written: {manifest}")


### 3. Understand the training and inference handoff

During training, paired RNA/protein graphs produce two latent representations. During inference, only
the RNA graph is encoded; its latent representation is sent through the trained protein decoder.

`RNA graph → RNA encoder → shared latent → protein decoder → predicted proteins`


### Check

Both graphs must have the same node count and ordering in paired training data. Feature counts differ:
RNA nodes carry genes and protein nodes carry ADTs. Edge arrays use integer node positions and have
shape `2 × number_of_edges`.


#### Why does this RNA graph have 17,434 features while the pretrained model expects 11,535?

The two counts come from two different upstream DGAT training workflows:

- **17,434 genes:** `Demo1_Train.ipynb` trains on **Tonsil alone**. After
  `preprocess_train_list` applies CytAssist QC, 17,434 Tonsil genes remain. This tutorial uses that
  same single-sample preprocessing to demonstrate how a paired RNA/protein training graph is built.
- **11,535 genes:** `Pretrain_DGAT.ipynb` creates the public pretrained model from **six paired
  datasets**: Tonsil, Tonsil AddOns, Breast, Glioblastoma, PBC-PR_6835-5A, and PBC_PR_6837.
  `preprocess_train_list` first finds genes shared by the input datasets, applies the 2.5% gene
  prevalence and spot-level QC within each dataset (while retaining protein-encoding genes), and then
  intersects the post-QC gene sets again. That final sorted intersection contains 11,535 genes and is
  saved upstream as `common_gene_11535.txt`. The 31-protein panel is derived analogously as the common
  post-QC protein set.

Consequently, 11,535 is not an arbitrary truncation of the Tonsil matrix: it is the cross-dataset
feature vocabulary on which the public checkpoint was trained, and it fixes the RNA encoder's input
width and gene order. A 17,434-feature matrix cannot be passed directly to that encoder. During real
pretrained inference, `fill_genes` selects and reorders the RNA matrix to `common_gene_11535.txt` and
inserts zeros for missing panel genes; `preprocess_ST` then normalizes it before graph construction.
Thus, the graph above illustrates the one-sample paired-training workflow, not the exact matrix supplied
to the six-dataset public checkpoint.


## Session 2 · Part 2 — Create every DGAT model component

**Goal:** understand and instantiate the four modules that are trained jointly. This lesson follows the
official `Model/dgat.py`: separate RNA and protein graph-attention encoders, an RNA decoder, and a
branched protein decoder. The notebook summarizes the architecture and creates a figure without loading
the full training environment.


### 1. Set dimensions from the processed data


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

from dgat_tutorial.data import load_tutorial_data
from dgat_tutorial.teaching import (
    DGAT_PRETRAINED_GENE_COUNT,
    DGAT_PRETRAINED_PROTEIN_COUNT,
    official_dgat_component_table,
)

dataset = load_tutorial_data(paths.raw_data)
# Teaching dimensions follow the public pretrained ST checkpoint (Demo3 uses 11535 genes /
# 31 proteins). The Tonsil AnnData has a broader gene panel; inference zero-fills to the
# common-gene list via official fill_genes before protein_predict.
gene_list_path = paths.root / "external" / "DGAT" / "resources" / f"common_gene_{DGAT_PRETRAINED_GENE_COUNT}.txt"
protein_list_path = paths.root / "external" / "DGAT" / "resources" / f"common_protein_{DGAT_PRETRAINED_PROTEIN_COUNT}.txt"
if gene_list_path.exists() and protein_list_path.exists():
    common_genes = [line.strip() for line in gene_list_path.read_text().splitlines() if line.strip()]
    common_proteins = [line.strip() for line in protein_list_path.read_text().splitlines() if line.strip()]
    print(f"Using official common lists: {len(common_genes)} genes, {len(common_proteins)} proteins")
else:
    common_genes = list(dataset.transcripts.columns)[:DGAT_PRETRAINED_GENE_COUNT]
    common_proteins = list(dataset.proteins.columns)[:DGAT_PRETRAINED_PROTEIN_COUNT]
    print(
        "Official common_gene/protein lists not found under external/DGAT/resources/; "
        f"using the first {len(common_genes)} genes / {len(common_proteins)} proteins from Tonsil for the architecture table."
    )
HIDDEN_DIM = 1024  # official Train_and_Predict.py
component_table = official_dgat_component_table(len(common_genes), common_proteins, HIDDEN_DIM)
component_table


### 2. Read the architecture from left to right

1. Each encoder performs three graph-attention stages with skip projections and LayerNorm.
2. After the first GAT stage, 16 feature-attention heads learn channel gates and their mean reweights the
   2048-dimensional representation.
3. Both encoders end in the same latent dimension so paired RNA and protein spots can be aligned.
4. The protein decoder shares two layers, then uses one output branch per protein. This lets proteins
   share signal while retaining protein-specific prediction heads.


#### Figure 6 — DGAT architecture and inference path


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.set_xlim(0, 12); ax.set_ylim(0, 6); ax.axis("off")

def box(x, y, w, h, label, color):
    patch = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.05", facecolor=color, edgecolor="#333333")
    ax.add_patch(patch); ax.text(x + w/2, y + h/2, label, ha="center", va="center", fontsize=9)
def arrow(x1, y1, x2, y2, style="-"):
    ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle="->", mutation_scale=12,
                                 linewidth=1.2, linestyle=style, color="#333333"))

box(0.3, 4.1, 1.6, 1.0, "RNA graph\nX_RNA, E_RNA", "#d9eaf7")
box(2.5, 4.1, 2.0, 1.0, "RNA GAT encoder", "#96c5e8")
box(5.2, 4.1, 1.6, 1.0, "z_RNA", "#e4d7f4")
box(0.3, 1.0, 1.6, 1.0, "Protein graph\nX_P, E_P", "#fde2cd")
box(2.5, 1.0, 2.0, 1.0, "Protein GAT encoder", "#f5b97f")
box(5.2, 1.0, 1.6, 1.0, "z_protein", "#e4d7f4")
box(8.0, 4.1, 1.6, 1.0, "RNA decoder", "#d7ecd9")
box(8.0, 1.0, 1.6, 1.0, "Protein decoder", "#d7ecd9")
box(10.2, 4.1, 1.5, 1.0, "RNA output", "#eeeeee")
box(10.2, 1.0, 1.5, 1.0, "Protein output", "#eeeeee")
for start, end in [((1.9,4.6),(2.5,4.6)),((4.5,4.6),(5.2,4.6)),((1.9,1.5),(2.5,1.5)),
                   ((4.5,1.5),(5.2,1.5)),((6.8,4.6),(8.0,4.6)),((9.6,4.6),(10.2,4.6)),
                   ((6.8,1.5),(8.0,1.5)),((9.6,1.5),(10.2,1.5))]: arrow(*start,*end)
arrow(6.8, 4.4, 8.0, 1.8, "--")
arrow(6.8, 1.7, 8.0, 4.3, "--")
ax.text(7.25, 3.0, "cross-modal\nprediction", ha="center", va="center", fontsize=9)
ax.text(6.0, 3.0, "latent alignment", ha="center", va="center", fontsize=9, rotation=90)
ax.plot([6.0,6.0],[2.0,4.1], color="#6b4c9a", linestyle=":", linewidth=2)
architecture_path = paths.figures / "session02_dgat_architecture.png"
fig.savefig(architecture_path, dpi=180, bbox_inches="tight")
plt.show()


In [ ]:
component_path = paths.results / "session02_dgat_components.csv"
component_table.to_csv(component_path, index=False)
manifest = write_checkpoint(
    "2.2", [component_path, architecture_path],
    summary={"modules": len(component_table)}, start=paths.root,
)
print(f"Checkpoint written: {manifest}")


### Check

Point to the exact inference route in the figure: **RNA graph → RNA encoder → z_RNA → protein
decoder**. The protein encoder and RNA decoder are training-time partners that make the shared latent
space learnable; they are not needed to impute protein on an RNA-only sample.


## Session 2 · Part 3 — Training objective (discussion; training skipped)

**Goal:** connect the four modules to five losses, four optimizers, backpropagation, evaluation, and
checkpoints — conceptually. **This tutorial does not train DGAT** (Colab / workshop compute limits).
We inspect the official objective and a pedagogical training step, then continue to pretrained
predictions in Part 4. Full training belongs in the upstream
[DGAT repository](https://github.com/osmanbeyoglulab/DGAT), not the live session.


### 1. Decompose the training objective


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

from dgat_tutorial.teaching import (
    DGAT_TRAIN_LOSS_WEIGHTS,
    official_dgat_loss_table,
    official_dgat_optimizer_table,
    weighted_training_objective,
)

loss_table = official_dgat_loss_table()
display(loss_table)
display(official_dgat_optimizer_table())
print(f"DGAT training coefficients (α,β,γ,δ,η): {DGAT_TRAIN_LOSS_WEIGHTS}")


The DGAT **training loop** uses five coefficients:

$$
(\alpha,\beta,\gamma,\delta,\eta)=(5,1,1,3,1).
$$

Its scalar training objective is

$$
\begin{aligned}
\mathcal{L}_{\mathrm{train}} ={}&
5\,\mathcal{L}_{\mathrm{RNA\ recon}}
+ \widetilde{\beta}\,\mathcal{L}_{\mathrm{protein\ recon}}
+ \widetilde{\gamma}\,\mathcal{L}_{\mathrm{alignment}} \\
&+ \widetilde{\delta}\,\mathcal{L}_{\mathrm{RNA}\rightarrow\mathrm{protein}}
+ 1\,\mathcal{L}_{\mathrm{protein}\rightarrow\mathrm{RNA}},
\end{aligned}
$$

Each corresponding
effective coefficient—$\widetilde{\beta}$, $\widetilde{\gamma}$, or
$\widetilde{\delta}$—is replaced by zero when its unweighted loss is below $0.015$.
The RNA-reconstruction coefficient $\alpha=5$ and protein→RNA coefficient $\eta=1$
are not subject to this soft-zero rule.

The RNA→protein term directly trains the path used at inference. Always log the components
separately—a falling total can hide a failing task.


In [ ]:
example_logged_terms = dict(
    rna_reconstruction=0.42, protein_reconstruction=0.31, latent_alignment=0.08,
    protein_prediction=0.37, rna_prediction=0.46,
)
example_total = weighted_training_objective(**example_logged_terms)
print("Actual upstream training calculation for these example loss values:")
print("5×0.42 + 1×0.31 + 1×0.08 + 3×0.37 + 1×0.46")
print(f"Total training loss = {example_total:.2f}")
print("All five example losses exceed 0.015, so no coefficient is soft-zeroed.")


### 2. Inspect one real training step (pedagogical mirror)

This cell is for reading and discussion. It is **not** a complete training loop: it does not load
batches, run epochs, validate on a held-out sample, or save checkpoints. The figure below contrasts
the **recommended** held-out validate/save-best workflow (dashed) with what upstream
`Train_and_Predict.train(...)` currently exposes (solid boxes only through backprop + EB early stop).


In [ ]:
try:
    import torch
except ImportError:
    torch = None

from dgat_tutorial.teaching import DGAT_LOSS_SOFT_THRESHOLD, DGAT_TRAIN_LOSS_WEIGHTS, apply_soft_loss_weights

def dgat_training_step(batch, modules, optimizers, rmse_loss, mse_loss, weights=DGAT_TRAIN_LOSS_WEIGHTS):
    # Readable mirror of the upstream DGAT optimization step (not invoked in the tutorial path).
    if torch is None:
        raise ImportError("torch is required to execute dgat_training_step; use eccb-dgat-official.")
    encoder_rna, decoder_rna, encoder_protein, decoder_protein = modules
    for optimizer in optimizers:
        optimizer.zero_grad()

    x_rna = batch["mRNA"].x
    e_rna = batch[("mRNA", "mRNA_knn", "mRNA")].edge_index
    x_protein = batch["protein"].x
    e_protein = batch[("protein", "protein_knn", "protein")].edge_index

    z_rna = encoder_rna(x_rna, e_rna)
    z_protein = encoder_protein(x_protein, e_protein)

    losses = {
        "rna_reconstruction": rmse_loss(decoder_rna(z_rna), x_rna),
        "protein_reconstruction": rmse_loss(decoder_protein(z_protein), x_protein),
        "latent_alignment": mse_loss(z_rna, z_protein),
        "protein_prediction": rmse_loss(decoder_protein(z_rna), x_protein),
        "rna_prediction": rmse_loss(decoder_rna(z_protein), x_rna),
    }
    effective = apply_soft_loss_weights(
        float(losses["protein_reconstruction"].detach()),
        float(losses["latent_alignment"].detach()),
        float(losses["protein_prediction"].detach()),
        weights=weights,
        soft_threshold=DGAT_LOSS_SOFT_THRESHOLD,
    )
    total = sum(weight * loss for weight, loss in zip(effective, losses.values()))
    total.backward()
    for module in modules:
        torch.nn.utils.clip_grad_norm_(module.parameters(), max_norm=1.0)
    for optimizer in optimizers:
        optimizer.step()
    return {name: float(value.detach()) for name, value in losses.items()} | {"total": float(total.detach())}

print(f"torch={'unavailable' if torch is None else torch.__version__}")
print(f"Default train weights α,β,γ,δ,η = {DGAT_TRAIN_LOSS_WEIGHTS}; soft-threshold = {DGAT_LOSS_SOFT_THRESHOLD}")
print("dgat_training_step is defined for inspection only; the tutorial does not call it.")


#### Figure 7 — Upstream training path vs recommended validation loop


In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.6)); ax.set_xlim(0, 12); ax.set_ylim(0, 3.4); ax.axis("off")
labels = ["paired samples", "normalize + graphs", "forward pass", "5 losses", "backprop + clip", "EB early stop"]
colors = ["#d9eaf7", "#d9eaf7", "#e4d7f4", "#fde2cd", "#f5b97f", "#d7ecd9"]
xs = [0.2, 2.1, 4.0, 5.9, 7.8, 9.7]
for x, label, color in zip(xs, labels, colors):
    patch = FancyBboxPatch((x, 1.35), 1.55, 0.9, boxstyle="round,pad=0.04", facecolor=color, edgecolor="#333")
    ax.add_patch(patch); ax.text(x+0.775, 1.8, label, ha="center", va="center", fontsize=8.5)
for left in xs[:-1]:
    ax.add_patch(FancyArrowPatch((left+1.55,1.8),(left+1.9,1.8),arrowstyle="->",mutation_scale=11))
# Recommended but not implemented in upstream train()
for x, label in [(7.8, "held-out validate"), (9.7, "save best")]:
    patch = FancyBboxPatch((x, 0.15), 1.55, 0.75, boxstyle="round,pad=0.04", facecolor="#f7f7f7",
                           edgecolor="#666", linestyle="--")
    ax.add_patch(patch); ax.text(x+0.775, 0.525, label, ha="center", va="center", fontsize=8, color="#444")
ax.text(6.0, 3.15, "solid = upstream train(); dashed = recommended publication workflow (not in train())",
        ha="center", fontsize=9)
workflow_path = paths.figures / "session02_training_workflow.png"
fig.savefig(workflow_path, dpi=180, bbox_inches="tight"); plt.show()


### 3. Training is skipped in this tutorial

Colab and the live workshop do **not** launch `Train_and_Predict.train(...)`. Reasons:

- official training needs multi-sample CITE-seq assets, substantial CPU/GPU time, and the separate
  DGAT dependency stack;
- the participant path evaluates **verified pretrained** Tonsil predictions instead.

Organizers who need to reproduce weights should use the upstream DGAT demos outside this tutorial.


In [ ]:
print("Full DGAT training is skipped in the Colab / workshop path.")
print("Continue to Part 4 to load verified pretrained Tonsil predictions.")


### 4. What a trustworthy training run must save (for later study)

If you train DGAT later, save train/validation sample IDs, common gene/protein lists, preprocessing
parameters, random seed, per-epoch component losses, validation correlations, and all four state
dictionaries. Split by biological sample—not random spots—to avoid spatial and donor leakage. This
tutorial's committed predictions remain separate from observed evaluation proteins.


In [ ]:
loss_path = paths.results / "session02_dgat_loss_terms.csv"
loss_table.to_csv(loss_path, index=False)
manifest = write_checkpoint(
    "2.3", [loss_path, workflow_path],
    summary={"loss_terms": len(loss_table), "full_training_executed": False, "runtime": "colab_skip_training"},
    start=paths.root,
)
print(f"Checkpoint written: {manifest}")


### Check

Before moving on, explain why validation must hold out whole samples, which loss directly supervises
protein imputation, and which two modules are needed for RNA-only inference.


## Session 2 · Part 4 — Load and validate pretrained predictions

**Goal:** use a verified DGAT artifact after learning how it was built and trained. Conference laptops
load committed official predictions for speed and reproducibility.


### 1. Inspect prediction provenance before trusting values


In [ ]:
from dgat_tutorial.dgat import load_prediction_metadata, load_prediction_table, write_prediction_artifact

source_path = paths.raw_data / "dgat_predictions.csv"
if not source_path.is_file():
    raise FileNotFoundError(f"Missing verified prediction table: {source_path}")
predicted_proteins = load_prediction_table(str(source_path))
metadata = load_prediction_metadata(source_path)
if metadata is None:
    raise FileNotFoundError(f"Missing provenance sidecar for {source_path}")
display(metadata, predicted_proteins.head())


### 2. The inference operation represented by this artifact

```python
z_rna = encoder_rna(x_rna, rna_edge_index)
predicted_protein = decoder_protein(z_rna)
```

The distributed artifact was generated with the upstream DGAT inference workflow and is accompanied by
provenance metadata. Participants inspect that metadata before using the values.


In [ ]:
output_path = paths.processed_data / "predicted_proteins.csv"
metadata_path = write_prediction_artifact(
    predicted_proteins, output_path,
    method=str(metadata["method"]), source=str(metadata["source"]),
    evaluation_note=str(metadata["evaluation_note"]),
)
manifest = write_checkpoint(
    "2.4", [output_path, metadata_path],
    summary={"spots": len(predicted_proteins), "proteins": predicted_proteins.shape[1]}, start=paths.root,
)
print(f"Checkpoint written: {manifest}")


### Check

Do not evaluate a model on protein values that were used to train or select it. Read the sidecar's
evaluation note and confirm that spot IDs and protein names match the target dataset.


## Session 2 · Part 5 — Visualize inferred protein landscapes

**Goal:** inspect prediction distributions and spatial patterns before calculating accuracy metrics.
A plausible-looking map is not proof of correctness; it is a diagnostic for range compression, isolated
artifacts, tissue-edge effects, and expected regional structure.


### 1. Align predictions to spatial coordinates


In [ ]:
import matplotlib.pyplot as plt

from dgat_tutorial.checkpoints import preferred_prediction_path
from dgat_tutorial.data import load_tutorial_data
from dgat_tutorial.dgat import load_prediction_table
from dgat_tutorial.plotting import plot_spatial_feature

dataset = load_tutorial_data(paths.raw_data)
prediction_path = preferred_prediction_path(paths)
predicted = load_prediction_table(str(prediction_path))
common_spots = dataset.spots.index.intersection(predicted.index)
if common_spots.empty:
    raise ValueError("Spatial data and predictions have no shared spot IDs; check that the assets match.")
spots = dataset.spots.loc[common_spots]
predicted = predicted.loc[common_spots]
print(f"Aligned {len(common_spots)} spots and {predicted.shape[1]} predicted proteins")


#### Figure 8 — Prediction distributions


In [ ]:
# Prefer robust IQR over variance so a few outlier spots do not dominate feature selection.
iqr = predicted.quantile(0.75) - predicted.quantile(0.25)
proteins_to_plot = list(iqr.nlargest(min(4, predicted.shape[1])).index)
fig, axes = plt.subplots(1, len(proteins_to_plot), figsize=(3.2 * len(proteins_to_plot), 3.2))
axes = [axes] if len(proteins_to_plot) == 1 else axes
for ax, protein in zip(axes, proteins_to_plot):
    ax.hist(predicted[protein], bins=30, color="#4c78a8")
    ax.set_title(protein); ax.set_xlabel("predicted abundance"); ax.set_ylabel("spots")
distribution_path = paths.figures / "session02_prediction_distributions.png"
fig.tight_layout(); fig.savefig(distribution_path, dpi=160, bbox_inches="tight"); plt.show()


#### Figure 9 — Predicted spatial protein maps


In [ ]:
n_cols = 2
n_rows = (len(proteins_to_plot) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(9, 4 * n_rows), squeeze=False)
# Shared color scale across the selected panel for fair visual comparison.
shared_vmin = float(predicted[proteins_to_plot].min().min())
shared_vmax = float(predicted[proteins_to_plot].max().max())
for ax, protein in zip(axes.ravel(), proteins_to_plot):
    plot_spatial_feature(
        spots, predicted[protein], f"Predicted {protein}", ax=ax, vmin=shared_vmin, vmax=shared_vmax
    )
for ax in axes.ravel()[len(proteins_to_plot):]: ax.axis("off")
spatial_path = paths.figures / "session02_predicted_protein_maps.png"
fig.tight_layout(); fig.savefig(spatial_path, dpi=160, bbox_inches="tight"); plt.show()


In [ ]:
manifest = write_checkpoint(
    "2.5", [prediction_path, distribution_path, spatial_path],
    summary={"spots": len(common_spots), "proteins_plotted": proteins_to_plot}, start=paths.root,
)
print(f"Checkpoint written: {manifest}")


### Check

Identify one map worth evaluating and one possible artifact. Then continue to Session 3, where observed
proteins are used for pointwise correlation, spatial coherence, and side-by-side landscape comparison.
